# Module 7: Agent-as-Tool: Deploy to Amazon Bedrock AgentCore

**Pattern 5: Agent-as-Tool**: Orchestrator delegates to researcher, analyzer, and synthesizer specialists wrapped as `@tool` functions.

![Agent-as-Tool Runtime architecture](./architecture.png)

**What this notebook covers:**
1. Install the AgentCore CLI
2. Configure and deploy
3. Review `main.py`
4. Find the Runtime ARN
5. Invoke the deployed agent with boto3
6. Observability in CloudWatch
7. Cleanup

---

## Step 1: Install the AgentCore CLI

In [ ]:
!uv pip install -q bedrock-agentcore-starter-toolkit bedrock-agentcore strands-agents boto3

---

## Step 2: Configure and deploy

Run the following commands in a **terminal** from the `production/` folder.

```bash
# Navigate to this folder
cd samples/07-agent-as-tool/production

# Configure the agent for deployment
agentcore configure -e main.py
# When prompted for a name, enter one ≤ 23 characters, e.g.: agenttool

# Deploy (packages code to S3, provisions Runtime via CloudFormation: ~3-5 min)
agentcore deploy
```

When the deploy finishes, the output includes the **Runtime ARN**: copy it for Step 4.

---

## Step 3: Review `main.py`

This is the code that runs inside the Runtime container.

In [ ]:
print(open("main.py").read())

---

## Step 4: Find the Runtime ARN

Run this cell to list all deployed Runtimes and find the ARN for this module.

In [ ]:
import boto3

client_control = boto3.client("bedrock-agentcore-control", region_name="us-east-1")
response = client_control.list_agent_runtimes()

print(f"{'Name':<40} {'ARN'}")
print("-" * 100)
for rt in response.get("agentRuntimes", []):
    print(f"{rt['agentRuntimeName']:<40} {rt['agentRuntimeArn']}")

---

## Step 5: Invoke the deployed agent

Paste the ARN from Step 4 into `RUNTIME_ARN` below, then run the cell.

In [ ]:
import boto3, json, uuid
from botocore.config import Config

RUNTIME_ARN = ""  # paste ARN from Step 4 here

if not RUNTIME_ARN:
    raise ValueError("Set RUNTIME_ARN above before running this cell.")

client = boto3.client(
    "bedrock-agentcore",
    region_name="us-east-1",
    config=Config(read_timeout=300),   # pipelines can take 60-180s
)

response = client.invoke_agent_runtime(
    agentRuntimeArn=RUNTIME_ARN,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({
        "prompt": "NovaCart Premium Tier: Options A ($19.99/mo invite-only), B ($14.99/mo 5% pilot), C ($12.99/mo full launch). Target: +15% CLV in 6 months."
    }).encode(),
    qualifier="DEFAULT",
)

result = json.loads(response["response"].read())
print(result.get("response", result))

---

## Step 6: Observability

After invoking, traces appear in **CloudWatch > X-Ray > Traces** or **Amazon Bedrock > AgentCore > Observability**.

What you see per invocation:
- Root span per `invoke_agent_runtime` call
- Child span per `Agent()` call inside the pipeline
- Tool call spans nested under each agent
- Duration breakdown per stage

No extra configuration needed: `aws-opentelemetry-distro` is in `requirements.txt` and AgentCore installs it automatically.

---

## Step 7: Cleanup

Two steps: both required:

```bash
# Step 1: Reset local config (does NOT delete AWS resources)
agentcore remove all -y

# Step 2: Delete the Runtime from AWS
agentcore deploy
```

Verify cleanup:

```bash
aws bedrock-agentcore list-agent-runtimes --region us-east-1
```